In [28]:
import arcpy
from pathlib import Path
import os
import shutil
import pandas as pd
from datetime import datetime
import re
import unicodedata


In [29]:
# =============================================================================
# PATHS BASE - CL_MLP_PAO
# =============================================================================

# Carpeta de entrada donde se dejan los vuelos de drone sin procesar
PATH_INPUT_VUELOS_DRONE = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"


# =============================================================================
# PROYECTO ARCGIS PRO
# =============================================================================

# Proyecto APRX del Visor Territorial SIG PAO
PATH_APRX_VISOR_TERRITORIAL = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"


# =============================================================================
# GEODATABASES
# =============================================================================

# Geodatabase de imágenes PAO
PATH_GDB_IMAGENES = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"

# Geodatabase principal PAO v1
PATH_GDB_PAO_V1 = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"


# =============================================================================
# FEATURE CLASSES - IMÁGENES
# =============================================================================

# Feature class de imágenes oblicuas de drone
PATH_FC_IMAGENES_OBLICUAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"
    r"\CL_MLP_PAO_01_Imagenes_Oblicuas\CL_MLP_PAO_Oblicuas_Drone_v2"
)

# Índice / diccionario de vuelos PAO para imágenes
PATH_FC_INDICE_VUELOS_IMGS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
)


# =============================================================================
# FEATURE CLASSES - COMPLEMENTOS
# =============================================================================

# Feature class de macrozonas PAO
PATH_FC_MACROZONAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_ZONAS_Macrozonas_PO"
)


# =============================================================================
# FEATURE CLASSES - VIDEOS
# =============================================================================

# Feature class de videos de drone en terreno
PATH_FC_VIDEOS_DRONE = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_07_IMAGENES_TERRENO\MLP_SIG_PAO_Videos_Drone"
)


# =============================================================================
# CARPETAS DE VUELOS PROCESADOS / FECHA 26_06
# =============================================================================

# Carpeta de vuelos Drone - Chacay
PATH_DRONE_CHACAY_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_Drone\26_06"

# Carpeta de vuelos Drone - Chacay El Mauro
PATH_DRONE_CHACAY_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro
PATH_DRONE_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro Puerto Punta Chungo
PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Puerto_Punta_Chungo_Drone\26_06"

# Carpeta de vuelos Drone - Puerto Punta Chungo
PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Puerto_Punta_Chungo_Drone\26_06"


# =============================================================================
# LISTAS AGRUPADAS
# =============================================================================

# Lista de carpetas de vuelos por zona
PATHS_CARPETAS_VUELOS_DRONE = [
    PATH_DRONE_CHACAY_2606,
    PATH_DRONE_CHACAY_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606,
    PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606,
]

# Diccionario general de paths principales
PATHS_PAO = {
    "input_vuelos_drone": PATH_INPUT_VUELOS_DRONE,
    "aprx_visor_territorial": PATH_APRX_VISOR_TERRITORIAL,
    "gdb_imagenes": PATH_GDB_IMAGENES,
    "gdb_pao_v1": PATH_GDB_PAO_V1,
    "fc_imagenes_oblicuas": PATH_FC_IMAGENES_OBLICUAS,
    "fc_indice_vuelos_imgs": PATH_FC_INDICE_VUELOS_IMGS,
    "fc_macrozonas": PATH_FC_MACROZONAS,
    "fc_videos_drone": PATH_FC_VIDEOS_DRONE,
    "carpetas_vuelos_drone": PATHS_CARPETAS_VUELOS_DRONE,
}

In [30]:
# =============================================================================
# FASE 1 - CONFIGURACION DE CARGA DE IMAGENES
# =============================================================================

# Carpeta oficial donde llegan los vuelos drone que deben evaluarse para cargar al mosaico.
PATH_INPUT_SCAN = Path(PATH_INPUT_VUELOS_DRONE)

IMAGE_EXTENSIONS = {
    ".tif",
    ".tiff",
    ".jpg",
    ".jpeg",
    ".png",
    ".sid",
    ".jp2",
    ".ecw",
}

PATH_INPUT_SCAN

WindowsPath('//amssclgis10.ams.gmams.cl/CL_MLP_PAO/Vuelos_Drone_Sin_Procesar/INPUT')

## Fase 1.1 - Buscar imagenes nuevas en carpeta de vuelos drone


In [31]:
def scan_input_images(input_folder, extensions=None):
    input_folder = Path(input_folder)
    extensions = {ext.lower() for ext in (extensions or IMAGE_EXTENSIONS)}

    if not input_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta input: {input_folder}")

    rows = []

    for file_path in sorted(input_folder.rglob("*")):
        if not file_path.is_file() or file_path.suffix.lower() not in extensions:
            continue

        stat = file_path.stat()
        rows.append(
            {
                "file_name": file_path.name,
                "stem": file_path.stem,
                "extension": file_path.suffix.lower(),
                "path": str(file_path),
                "relative_path": str(file_path.relative_to(input_folder)),
                "size_mb": round(stat.st_size / (1024 * 1024), 3),
                "modified_at": datetime.fromtimestamp(stat.st_mtime),
            }
        )

    return pd.DataFrame(rows)


input_images_df = scan_input_images(PATH_INPUT_SCAN)
print(f"Carpeta evaluada: {PATH_INPUT_SCAN}")
print(f"Imagenes encontradas en entrada de vuelos drone: {len(input_images_df)}")
input_images_df.head(20)

Carpeta evaluada: \\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT
Imagenes encontradas en entrada de vuelos drone: 1999


,file_name,stem,extension,path,relative_path,size_mb,modified_at
0,DJI_20260529101422_0002_V.JPG,DJI_20260529101422_0002_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.959,2026-06-11 10:59:23
1,DJI_20260529101441_0004_V.JPG,DJI_20260529101441_0004_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.956,2026-06-11 10:59:23
2,DJI_20260529101449_0005_V.JPG,DJI_20260529101449_0005_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.017,2026-06-11 10:59:23
3,DJI_20260529101506_0006_V.JPG,DJI_20260529101506_0006_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.933,2026-06-11 10:59:23
4,DJI_20260529101533_0007_V.JPG,DJI_20260529101533_0007_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.985,2026-06-11 10:59:23
5,DJI_20260529101603_0008_V.JPG,DJI_20260529101603_0008_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.170,2026-06-11 10:59:23
6,DJI_20250820094414_0002_V.JPG,DJI_20250820094414_0002_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.157,2026-06-11 10:59:24
7,DJI_20250820094426_0004_V.JPG,DJI_20250820094426_0004_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.183,2026-06-11 10:59:24
8,DJI_20250820094432_0005_V.JPG,DJI_20250820094432_0005_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.215,2026-06-11 10:59:24
9,DJI_20250820094437_0006_V.JPG,DJI_20250820094437_0006_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.188,2026-06-11 10:59:25


## Fase 1.2 - Revisar campos del mosaic dataset


In [32]:
def list_dataset_fields(dataset_path):
    fields = arcpy.ListFields(dataset_path)
    return pd.DataFrame(
        [
            {
                "name": field.name,
                "alias": field.aliasName,
                "type": field.type,
                "length": field.length,
                "required": field.required,
                "editable": field.editable,
            }
            for field in fields
        ]
    )


mosaic_fields_df = list_dataset_fields(PATH_MOSAIC_DATASET)
print(f"Campos del mosaic dataset: {len(mosaic_fields_df)}")
mosaic_fields_df

Campos del mosaic dataset: 30


,name,alias,type,length,required,editable
0,OBJECTID,OBJECTID,OID,4,True,False
1,Name,Name,String,200,False,True
2,MinPS,MinPS,Double,8,True,True
3,MaxPS,MaxPS,Double,8,True,True
4,LowPS,LowPS,Double,8,True,True
5,HighPS,HighPS,Double,8,True,True
6,Category,Category,Integer,4,True,True
7,Tag,Tag,String,100,False,True
8,GroupName,GroupName,String,100,False,True
9,ProductName,ProductName,String,100,False,True


## Fase 1.3 - Crear DataFrame del mosaic dataset


In [33]:
def detect_candidate_path_fields(fields_df):
    tokens = ("path", "uri", "url", "file", "name", "source", "raster")
    candidate_fields = []

    for _, row in fields_df.iterrows():
        field_name = row["name"]
        field_type = row["type"]
        normalized_name = field_name.lower()

        if field_type in ("String", "Guid") and any(token in normalized_name for token in tokens):
            candidate_fields.append(field_name)

    return candidate_fields


def table_to_dataframe(dataset_path, fields=None, max_rows=5000):
    if fields is None:
        fields = [field.name for field in arcpy.ListFields(dataset_path) if field.type not in ("Geometry", "Raster", "Blob")]

    rows = []

    with arcpy.da.SearchCursor(dataset_path, fields) as cursor:
        for index, values in enumerate(cursor):
            if index >= max_rows:
                break

            rows.append(dict(zip(fields, values)))

    return pd.DataFrame(rows)


candidate_path_fields = detect_candidate_path_fields(mosaic_fields_df)
print("Campos candidatos para path/name del raster:", candidate_path_fields)

mosaic_df = table_to_dataframe(PATH_MOSAIC_DATASET, max_rows=5000)
print(f"Registros leidos del mosaic dataset: {len(mosaic_df)}")
mosaic_df.head(20)

Campos candidatos para path/name del raster: ['Name', 'GroupName', 'ProductName', 'UriHash', 'URL']
Registros leidos del mosaic dataset: 926


,OBJECTID,Name,MinPS,MaxPS,LowPS,HighPS,Category,Tag,GroupName,ProductName,...,Sector,Sensor,Proyecto,FechaAdqui,FechaCarga,Estado,URL,NombreVuelo,Shape.STArea(),Shape.STLength()
0,1779,CL_MLP_PAO_IF_Ortho_26_01_07_ED2,0.0,0.159038,0.015904,0.015904,1,Dataset,,1779,...,ED2,DJI MAVIC 3 ENTERPRISE,PAO,2026-01-07 04:00:00,2026-01-28,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,148_26_01_07_ED2,2375.039197,199.733182
1,17123,Ov_i9A2_L01_R0000FE03_C0000D54F,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,2475.953422,328.660039
2,17124,Ov_i9A2_L01_R0000FE03_C0000D550,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,1272.810296,225.131612
3,17125,Ov_i9A2_L01_R0000FE03_C0000D551,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,212.241920,128.369543
4,17126,Ov_i9A2_L01_R0000FE03_C0000D556,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,1471.408888,218.847932
5,17127,Ov_i9A2_L01_R0000FE03_C0000D557,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,4126.096991,287.850742
6,17128,Ov_i9A2_L01_R0000FE03_C0000D558,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,907.186032,148.854099
7,17136,Ov_i9A2_L01_R0000FE04_C0000D54F,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,9872.389728,397.439600
8,17137,Ov_i9A2_L01_R0000FE04_C0000D550,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,9872.379792,397.439400
9,17138,Ov_i9A2_L01_R0000FE04_C0000D551,0.0,0.028905,0.019406,0.019406,2,Dataset,,,...,None,None,None,NaT,NaT,None,None,None,9784.618551,393.186621


## Fase 1.4 - Comparar vuelos drone de entrada contra el mosaic dataset


In [34]:
def normalize_image_key(value):
    if value is None:
        return None

    value = str(value).strip().replace("/", "\\")

    if not value:
        return None

    return value.lower()


def extract_mosaic_path_parts(value):
    normalized_value = normalize_image_key(value)

    if not normalized_value:
        return {
            "mosaic_path": None,
            "mosaic_file_name": None,
            "mosaic_stem": None,
            "mosaic_key": None,
            "mosaic_file_key": None,
            "mosaic_stem_key": None,
        }

    mosaic_path = normalized_value

    if "file?id=" in mosaic_path:
        mosaic_path = mosaic_path.split("file?id=", 1)[1]

    if "&" in mosaic_path:
        mosaic_path = mosaic_path.split("&", 1)[0]

    mosaic_path = mosaic_path.strip()
    mosaic_file_name = re.split(r"[\\/]", mosaic_path)[-1] if mosaic_path else None
    mosaic_stem = Path(mosaic_file_name).stem if mosaic_file_name else None

    return {
        "mosaic_path": mosaic_path,
        "mosaic_file_name": mosaic_file_name,
        "mosaic_stem": mosaic_stem,
        "mosaic_key": mosaic_path,
        "mosaic_file_key": mosaic_file_name.lower() if mosaic_file_name else None,
        "mosaic_stem_key": mosaic_stem.lower() if mosaic_stem else None,
    }


def build_mosaic_image_inventory(mosaic_df, candidate_fields):
    inventory_rows = []

    for field in candidate_fields:
        if field not in mosaic_df.columns:
            continue

        for row_index, value in mosaic_df[field].dropna().items():
            path_parts = extract_mosaic_path_parts(value)

            if not path_parts["mosaic_key"]:
                continue

            inventory_rows.append(
                {
                    "mosaic_row_index": row_index,
                    "source_field": field,
                    "source_value": value,
                    **path_parts,
                }
            )

    inventory_df = pd.DataFrame(inventory_rows)

    if inventory_df.empty:
        return inventory_df

    return inventory_df.drop_duplicates(subset=["source_field", "mosaic_key", "mosaic_file_key", "mosaic_stem_key"])


def build_mosaic_lookup_from_inventory(mosaic_inventory_df):
    lookup = set()

    if mosaic_inventory_df.empty:
        return lookup

    for column in ["mosaic_key", "mosaic_file_key", "mosaic_stem_key"]:
        for value in mosaic_inventory_df[column].dropna():
            normalized_value = normalize_image_key(value)

            if normalized_value:
                lookup.add(normalized_value)

    return lookup


mosaic_image_inventory_df = build_mosaic_image_inventory(mosaic_df, candidate_path_fields)
mosaic_lookup = build_mosaic_lookup_from_inventory(mosaic_image_inventory_df)

print(f"Registros/path candidatos extraidos del mosaic dataset: {len(mosaic_image_inventory_df)}")
display(mosaic_image_inventory_df.head(50))

# Comparacion inicial usando nombres/path existentes del mosaic dataset.

if input_images_df.empty:
    input_images_df = input_images_df.assign(exists_in_mosaic=[], match_key=[])
else:
    input_images_df = input_images_df.copy()
    input_images_df["match_key"] = input_images_df["path"].map(normalize_image_key)
    input_images_df["exists_in_mosaic"] = input_images_df.apply(
        lambda row: any(
            key in mosaic_lookup
            for key in (
                normalize_image_key(row["path"]),
                normalize_image_key(row["file_name"]),
                normalize_image_key(row["stem"]),
            )
            if key
        ),
        axis=1,
    )

new_images_df = input_images_df[input_images_df["exists_in_mosaic"] == False].copy()

print(f"Imagenes en input: {len(input_images_df)}")
print(f"Imagenes ya detectadas en mosaic dataset: {int(input_images_df['exists_in_mosaic'].sum()) if not input_images_df.empty else 0}")
print(f"Imagenes nuevas candidatas a cargar: {len(new_images_df)}")

new_images_df

Registros/path candidatos extraidos del mosaic dataset: 2442


,mosaic_row_index,source_field,source_value,mosaic_path,mosaic_file_name,mosaic_stem,mosaic_key,mosaic_file_key,mosaic_stem_key
0,0,Name,CL_MLP_PAO_IF_Ortho_26_01_07_ED2,cl_mlp_pao_if_ortho_26_01_07_ed2,cl_mlp_pao_if_ortho_26_01_07_ed2,cl_mlp_pao_if_ortho_26_01_07_ed2,cl_mlp_pao_if_ortho_26_01_07_ed2,cl_mlp_pao_if_ortho_26_01_07_ed2,cl_mlp_pao_if_ortho_26_01_07_ed2
1,1,Name,Ov_i9A2_L01_R0000FE03_C0000D54F,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f,ov_i9a2_l01_r0000fe03_c0000d54f
2,2,Name,Ov_i9A2_L01_R0000FE03_C0000D550,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550,ov_i9a2_l01_r0000fe03_c0000d550
3,3,Name,Ov_i9A2_L01_R0000FE03_C0000D551,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551,ov_i9a2_l01_r0000fe03_c0000d551
4,4,Name,Ov_i9A2_L01_R0000FE03_C0000D556,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556,ov_i9a2_l01_r0000fe03_c0000d556
5,5,Name,Ov_i9A2_L01_R0000FE03_C0000D557,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557,ov_i9a2_l01_r0000fe03_c0000d557
6,6,Name,Ov_i9A2_L01_R0000FE03_C0000D558,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558,ov_i9a2_l01_r0000fe03_c0000d558
7,7,Name,Ov_i9A2_L01_R0000FE04_C0000D54F,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f,ov_i9a2_l01_r0000fe04_c0000d54f
8,8,Name,Ov_i9A2_L01_R0000FE04_C0000D550,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550,ov_i9a2_l01_r0000fe04_c0000d550
9,9,Name,Ov_i9A2_L01_R0000FE04_C0000D551,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551,ov_i9a2_l01_r0000fe04_c0000d551


Imagenes en input: 1999
Imagenes ya detectadas en mosaic dataset: 0
Imagenes nuevas candidatas a cargar: 1999


,file_name,stem,extension,path,relative_path,size_mb,modified_at,match_key,exists_in_mosaic
0,DJI_20260529101422_0002_V.JPG,DJI_20260529101422_0002_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.959,2026-06-11 10:59:23,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
1,DJI_20260529101441_0004_V.JPG,DJI_20260529101441_0004_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.956,2026-06-11 10:59:23,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
2,DJI_20260529101449_0005_V.JPG,DJI_20260529101449_0005_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,1.017,2026-06-11 10:59:23,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
3,DJI_20260529101506_0006_V.JPG,DJI_20260529101506_0006_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.933,2026-06-11 10:59:23,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
4,DJI_20260529101533_0007_V.JPG,DJI_20260529101533_0007_V,.jpg,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260206_Geosupport_primera entrega\SOLO_JPG_0...,0.985,2026-06-11 10:59:23,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
...,...,...,...,...,...,...,...,...,...
1994,GEOSP-TRN-002652_GS_ORTOFOTO_EV2_170526.tif,GEOSP-TRN-002652_GS_ORTOFOTO_EV2_170526,.tif,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,190.375,2026-06-05 11:48:27,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
1995,GEOSP-TRN-002655_GS_ORTOFOTO_EDT_17-05-2026.tif,GEOSP-TRN-002655_GS_ORTOFOTO_EDT_17-05-2026,.tif,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,21.961,2026-06-05 11:48:29,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
1996,GEOSP-TRN-002656_GD_ORTOFOTO_Tramo 2 linea 22 ...,GEOSP-TRN-002656_GD_ORTOFOTO_Tramo 2 linea 22 ...,.tif,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,650.561,2026-06-05 11:48:35,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False
1997,GEOSP-TRN-002660_GS_ORTOFOTO_EM1_150526.tif,GEOSP-TRN-002660_GS_ORTOFOTO_EM1_150526,.tif,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,138.719,2026-06-05 11:48:41,\\amssclgis10.ams.gmams.cl\cl_mlp_pao\vuelos_d...,False


## Fase 1.5 - Aplicar logica de renombre en Python

In [35]:
# Parametros equivalentes a la logica historica del Excel, ahora en Python.
RENAME_PREFIX = "CL_MLP_PAO_IF_Ortho"
DEFAULT_RENAMED_EXTENSION = ".tif"
ORTHO_MOSAIC_EXTENSIONS = {".tif", ".tiff"}

# Casos especiales conocidos. Agregar aqui excepciones cuando el nombre original no trae
# suficiente informacion para inferir el sector exacto usado en el mosaic dataset.
SECTOR_ALIASES = {
    "ESTACION DE VALVULAS N 2": "EV2",
    "ESTACION DE VALVULAS N2": "EV2",
    "ESTACION DE VALVULAS 2": "EV2",
    "ESTACION DISIPADORA TERMINAL": "EDT",
    "ACCESO A POZOS": "Acceso_a_pozos",
    "POZOS PRP": "Pozos_PRP",
    "CAMINO ALTERNATIVO SALAMANCA": "Camino_alternativo_Salamanca",
    "ESTACION DE BOMBEO N 2": "EB2",
    "ESTACION DE BOMBEO N 3": "EB3",
    "ESTACION CABECERA": "Estacion_Cabecera",
    "ESTACION CABECERAS": "Estacion_Cabecera",
    "ESTACION INTERMEDIA 3": "Estacion_Intermedia",
    "HELIPUERTO MAURO": "Helipuerto_Mauro",
    "PATIO 19B": "Patio_19B",
    "PATIO SDRNP": "Patio_SDRNP",
}

print(RENAME_PREFIX, DEFAULT_RENAMED_EXTENSION)

CL_MLP_PAO_IF_Ortho .tif


In [36]:
def strip_accents(value):
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(char for char in value if not unicodedata.combining(char))


def normalize_sector_text(value):
    value = strip_accents(value)
    value = value.replace("°", " ").replace("º", " ")
    value = re.sub(r"[^0-9A-Za-z]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def format_sector_token(value):
    value = normalize_sector_text(value)

    if not value:
        return None

    alias_key = value.upper()
    if alias_key in SECTOR_ALIASES:
        return SECTOR_ALIASES[alias_key]

    return value.replace(" ", "_")


def extract_date_token_from_filename(file_name):
    stem = Path(file_name).stem

    patterns = [
        r"(?P<year>20\d{2})(?P<month>\d{2})(?P<day>\d{2})",
        r"(?P<day>\d{2})[-_](?P<month>\d{2})[-_](?P<year>20\d{2})",
        r"(?P<day>\d{2})[-_](?P<month>\d{2})[-_](?P<year>\d{2})",
        r"(?P<day>\d{2})(?P<month>\d{2})(?P<year>\d{2})(?!\d)",
    ]

    for pattern in patterns:
        matches = list(re.finditer(pattern, stem))

        if not matches:
            continue

        match = matches[-1]
        year = match.group("year")
        year = year[-2:]
        month = match.group("month")
        day = match.group("day")

        return {
            "year": year,
            "month": month,
            "day": day,
            "date_token": f"{year}_{month}_{day}",
            "matched_text": match.group(0),
            "span": match.span(),
        }

    return None


def extract_sector_from_filename(file_name, date_match=None):
    stem = Path(file_name).stem
    working = stem

    if date_match:
        matched_text = date_match["matched_text"]
        working = working.replace(matched_text, " ")

    cleanup_patterns = [
        r"^GEOSP[-_ ]?TRN[-_ ]?\d+",
        r"\bGS\b",
        r"\bORTOFOTO\b",
        r"\bORTHOMOSAIC\b",
        r"\bORTOMOSAICO\b",
        r"\bDRONE\b",
        r"\bPAO\b",
        r"\(.*?\)",
    ]

    working = strip_accents(working).upper()

    for pattern in cleanup_patterns:
        working = re.sub(pattern, " ", working, flags=re.IGNORECASE)

    working = re.sub(r"[-_]+", " ", working)
    working = re.sub(r"\s+", " ", working).strip()

    return format_sector_token(working)


def build_expected_image_name(file_name, output_extension=DEFAULT_RENAMED_EXTENSION):
    date_match = extract_date_token_from_filename(file_name)

    if not date_match:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": None,
            "expected_sector": None,
            "rename_status": "sin_fecha",
        }

    sector = extract_sector_from_filename(file_name, date_match)

    if not sector:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": date_match["date_token"],
            "expected_sector": None,
            "rename_status": "sin_sector",
        }

    expected_stem = f"{RENAME_PREFIX}_{date_match['date_token']}_{sector}"
    expected_file_name = f"{expected_stem}{output_extension}"

    return {
        "expected_name": expected_stem,
        "expected_file_name": expected_file_name,
        "expected_stem": expected_stem,
        "expected_date_token": date_match["date_token"],
        "expected_sector": sector,
        "rename_status": "ok",
    }


def is_valid_date_parts(year, month, day):
    try:
        datetime(year=int(year), month=int(month), day=int(day))
        return True
    except ValueError:
        return False


def extract_date_token_from_filename(file_name):
    stem = Path(file_name).stem
    # No usar el ID GEOSP-TRN como fuente de fecha.
    stem_without_id = re.sub(r"^GEOSP[-_ ]?TRN[-_ ]?\d+", "", stem, flags=re.IGNORECASE)

    patterns = [
        r"(?<!\d)(?P<day>\d{2})[-_](?P<month>\d{2})[-_](?P<year>20\d{2}|\d{2})(?!\d)",
        r"(?<!\d)(?P<day>\d{2})(?P<month>\d{2})(?P<year>\d{2})(?!\d)",
        r"(?<!\d)(?P<year>20\d{2})(?P<month>\d{2})(?P<day>\d{2})(?!\d)",
    ]

    candidates = []
    for pattern in patterns:
        for match in re.finditer(pattern, stem_without_id):
            year = match.group("year")[-2:]
            month = match.group("month")
            day = match.group("day")

            if is_valid_date_parts(2000 + int(year), month, day):
                candidates.append(
                    {
                        "year": year,
                        "month": month,
                        "day": day,
                        "date_token": f"{year}_{month}_{day}",
                        "matched_text": match.group(0),
                        "span": match.span(),
                    }
                )

    return candidates[-1] if candidates else None


def normalize_sector_key(value):
    value = strip_accents(value).lower()
    value = re.sub(r"[^0-9a-z]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def extract_sector_candidates_from_filename(file_name, date_match=None):
    stem = Path(file_name).stem
    normalized_stem = strip_accents(stem)
    upper_stem = normalized_stem.upper()

    working = re.sub(r"^GEOSP[-_ ]?TRN[-_ ]?\d+", "", normalized_stem, flags=re.IGNORECASE)
    if date_match:
        working = working.replace(date_match["matched_text"], " ")

    working = re.sub(r"\(.*?\)", " ", working)
    working = re.sub(r"\[.*?\]", " ", working)
    raw_sector = normalize_sector_key(working)
    raw_sector = re.sub(r"(^|_)gs(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)gd(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)ortofoto(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)orthomosaic(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)ortomosaico(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)completa(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"(^|_)cortada(_|$)", "_", raw_sector)
    raw_sector = re.sub(r"_prioridad_\d+", "", raw_sector)
    raw_sector = re.sub(r"_+", "_", raw_sector).strip("_")

    candidates = []
    if raw_sector:
        candidates.append(raw_sector)

    alias_patterns = [
        (r"ESTACION DE BOMBEO.*N.?2", "eb2"),
        (r"ESTACION DE BOMBEO.*N.?3", "eb3"),
        (r"\bEB2\b", "eb2"),
        (r"\bEB3\b", "eb3"),
        (r"\bEV1\b", "ev1"),
        (r"\bEV2\b", "ev2"),
        (r"\bED1\b", "ed1"),
        (r"\bED2\b", "ed2"),
        (r"\bEDT\b", "edt"),
        (r"\bEBD\b", "ebd"),
        (r"\bEM2[-_ ]?2\b", "em2_s2"),
        (r"\bEM1\b", "em1"),
        (r"\bEM2\b", "em2"),
        (r"\bEM3\b", "em3"),
        (r"\bEM4\b", "em4"),
        (r"POZOS PRP", "pozos_prp"),
        (r"ACCESO.*POZOS", "acceso_a_pozos"),
        (r"CAMINO ALTERNATIVO SALAMANCA", "camino_alternativo_salamanca"),
        (r"ESTACION INTERMEDIA.?3", "estacion_intermedia"),
        (r"ESTACION CABECER", "estacion_cabecera"),
        (r"HELIPUERTO MAURO", "helipuerto_mauro"),
        (r"PATIO 19B", "patio_19b"),
        (r"PATIO SDRNP", "patio_sdrnp"),
        (r"DEPOSITO PATIO PULMON", "deposito_patio_pulmon"),
        (r"TORRES? E48.*E84", "torres_e48_a_e84"),
        (r"TORRES? E85.*E125|LINEA 33KV E.?085.*E.?125", "torres_e85_a_e_125"),
        (r"TORRES? E35.*E48|LINEA 22 KV E.?35.*E48", "torres_e35_a_e48"),
        (r"MONITOREO.*N.?2", "em2"),
        (r"SSEE", "subestacion-el-mauro_a_e35"),
    ]

    for pattern, sector in alias_patterns:
        if re.search(pattern, upper_stem):
            candidates.append(sector)

    return list(dict.fromkeys(candidates))


def build_expected_image_name(file_name, output_extension=DEFAULT_RENAMED_EXTENSION):
    if re.search(r"1001-03-T-CS|DW-", file_name, flags=re.IGNORECASE):
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": None,
            "expected_sector": None,
            "expected_sector_candidates": None,
            "rename_status": "descartar_posible_plano",
        }

    date_match = extract_date_token_from_filename(file_name)

    if not date_match:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": None,
            "expected_sector": None,
            "expected_sector_candidates": None,
            "rename_status": "sin_fecha",
        }

    sector_candidates = extract_sector_candidates_from_filename(file_name, date_match)

    if not sector_candidates:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": date_match["date_token"],
            "expected_sector": None,
            "expected_sector_candidates": None,
            "rename_status": "sin_sector",
        }

    expected_stems = [f"{RENAME_PREFIX}_{date_match['date_token']}_{sector}" for sector in sector_candidates]
    expected_stem = expected_stems[0]

    return {
        "expected_name": expected_stem,
        "expected_file_name": f"{expected_stem}{output_extension}",
        "expected_stem": expected_stem,
        "expected_date_token": date_match["date_token"],
        "expected_sector": sector_candidates[0],
        "expected_sector_candidates": "|".join(sector_candidates),
        "expected_stem_candidates": "|".join(expected_stems),
        "rename_status": "ok",
    }


if input_images_df.empty:
    input_expected_names_df = input_images_df.copy()
    ortho_input_images_df = input_images_df.copy()
    non_ortho_input_images_df = input_images_df.copy()
    for column in ["expected_name", "expected_file_name", "expected_stem", "expected_date_token", "expected_sector", "expected_sector_candidates", "expected_stem_candidates", "rename_status"]:
        input_expected_names_df[column] = pd.Series(dtype="object")
else:
    ortho_input_images_df = input_images_df[input_images_df["extension"].str.lower().isin(ORTHO_MOSAIC_EXTENSIONS)].copy()
    non_ortho_input_images_df = input_images_df[~input_images_df["extension"].str.lower().isin(ORTHO_MOSAIC_EXTENSIONS)].copy()
    expected_names_df = pd.DataFrame(
        [build_expected_image_name(file_name) for file_name in ortho_input_images_df["file_name"]]
    )
    input_expected_names_df = pd.concat([ortho_input_images_df.reset_index(drop=True), expected_names_df], axis=1)

print(f"Imagenes detectadas: {len(input_images_df)}")
print(f"Imagenes TIF/TIFF evaluadas para mosaico Ortho: {len(ortho_input_images_df)}")
print(f"Imagenes omitidas para este mosaico (JPG/otros): {len(non_ortho_input_images_df)}")
print(input_expected_names_df["rename_status"].value_counts(dropna=False))
input_expected_names_df[["file_name", "relative_path", "expected_file_name", "expected_name", "expected_date_token", "expected_sector", "expected_sector_candidates", "rename_status"]] if not input_expected_names_df.empty else input_expected_names_df

rename_status
ok    1999
Name: count, dtype: int64


,file_name,relative_path,expected_file_name,expected_name,expected_date_token,expected_sector,rename_status
0,DJI_20260529101422_0002_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101422_0002_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101422_0002_V,26_05_29,DJI_101422_0002_V,ok
1,DJI_20260529101441_0004_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101441_0004_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101441_0004_V,26_05_29,DJI_101441_0004_V,ok
2,DJI_20260529101449_0005_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101449_0005_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101449_0005_V,26_05_29,DJI_101449_0005_V,ok
3,DJI_20260529101506_0006_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101506_0006_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101506_0006_V,26_05_29,DJI_101506_0006_V,ok
4,DJI_20260529101533_0007_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101533_0007_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101533_0007_V,26_05_29,DJI_101533_0007_V,ok
...,...,...,...,...,...,...,...
1994,GEOSP-TRN-002652_GS_ORTOFOTO_EV2_170526.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EV2.tif,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EV2,26_05_17,GS_ORTOFOTO_EV2,ok
1995,GEOSP-TRN-002655_GS_ORTOFOTO_EDT_17-05-2026.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EDT.tif,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EDT,26_05_17,GS_ORTOFOTO_EDT,ok
1996,GEOSP-TRN-002656_GD_ORTOFOTO_Tramo 2 linea 22 ...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_14_GD_ORTOFOTO_TRAMO...,CL_MLP_PAO_IF_Ortho_26_05_14_GD_ORTOFOTO_TRAMO...,26_05_14,GD_ORTOFOTO_TRAMO_2_LINEA_22_KV_E_35_E48,ok
1997,GEOSP-TRN-002660_GS_ORTOFOTO_EM1_150526.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_15_GS_ORTOFOTO_EM1.tif,CL_MLP_PAO_IF_Ortho_26_05_15_GS_ORTOFOTO_EM1,26_05_15,GS_ORTOFOTO_EM1,ok


## Fase 1.6 - Comparar imagenes nuevas usando el nombre renombrado esperado

In [37]:
def add_python_rename_match(input_expected_names_df, mosaic_inventory_df):
    if input_expected_names_df.empty:
        result_df = input_expected_names_df.copy()
        result_df["expected_name_exists_in_mosaic"] = pd.Series(dtype="bool")
        result_df["load_status"] = pd.Series(dtype="object")
        return result_df

    result_df = input_expected_names_df.copy()
    match_lookup = {}
    if not mosaic_inventory_df.empty:
        for _, mosaic_row in mosaic_inventory_df.iterrows():
            for key_column in ["mosaic_key", "mosaic_file_key", "mosaic_stem_key"]:
                key = normalize_image_key(mosaic_row.get(key_column))

                if key and key not in match_lookup:
                    match_lookup[key] = mosaic_row

    def find_mosaic_match(row):
        keys = []
        for value in [row.get("expected_file_name"), row.get("expected_stem"), row.get("expected_name")]:
            keys.append(normalize_image_key(value))

        stem_candidates = row.get("expected_stem_candidates")
        if stem_candidates and not pd.isna(stem_candidates):
            for stem in str(stem_candidates).split("|"):
                keys.append(normalize_image_key(stem))
                keys.append(normalize_image_key(f"{stem}{DEFAULT_RENAMED_EXTENSION}"))

        for key in keys:
            if key and not pd.isna(key) and key in match_lookup:
                return match_lookup[key]

        return None

    result_df["mosaic_match"] = result_df.apply(find_mosaic_match, axis=1)
    result_df["expected_name_exists_in_mosaic"] = result_df["mosaic_match"].notna()
    result_df["matched_mosaic_source_field"] = result_df["mosaic_match"].map(lambda row: row.get("source_field") if row is not None else None)
    result_df["matched_mosaic_path"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_path") if row is not None else None)
    result_df["matched_mosaic_file_name"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_file_name") if row is not None else None)
    result_df["matched_mosaic_stem"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_stem") if row is not None else None)
    result_df = result_df.drop(columns=["mosaic_match"])
    result_df["load_status"] = result_df.apply(
        lambda row: row["rename_status"] if row["rename_status"] != "ok" else ("ya_cargada" if row["expected_name_exists_in_mosaic"] else "nueva_candidata"),
        axis=1,
    )

    return result_df


input_vs_mosaic_df = add_python_rename_match(input_expected_names_df, mosaic_image_inventory_df)
probable_new_tifs_df = input_vs_mosaic_df[input_vs_mosaic_df["load_status"] == "nueva_candidata"].copy()
review_or_discard_tifs_df = input_vs_mosaic_df[input_vs_mosaic_df["load_status"].isin(["sin_fecha", "sin_sector", "descartar_posible_plano"])].copy()

print(input_vs_mosaic_df["load_status"].value_counts(dropna=False) if not input_vs_mosaic_df.empty else "Sin imagenes en input")
print(f"TIF/TIFF probables nuevas para revisar/cargar: {len(probable_new_tifs_df)}")

review_columns = [
    "file_name",
    "relative_path",
    "expected_file_name",
    "expected_name",
    "expected_date_token",
    "expected_sector",
    "expected_sector_candidates",
    "rename_status",
    "expected_name_exists_in_mosaic",
    "matched_mosaic_source_field",
    "matched_mosaic_file_name",
    "matched_mosaic_path",
    "load_status",
]

input_vs_mosaic_df[review_columns] if not input_vs_mosaic_df.empty else input_vs_mosaic_df

load_status
nueva_candidata    1999
Name: count, dtype: int64


,file_name,relative_path,expected_file_name,expected_name,expected_date_token,expected_sector,rename_status,expected_name_exists_in_mosaic,matched_mosaic_source_field,matched_mosaic_file_name,matched_mosaic_path,load_status
0,DJI_20260529101422_0002_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101422_0002_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101422_0002_V,26_05_29,DJI_101422_0002_V,ok,False,None,None,None,nueva_candidata
1,DJI_20260529101441_0004_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101441_0004_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101441_0004_V,26_05_29,DJI_101441_0004_V,ok,False,None,None,None,nueva_candidata
2,DJI_20260529101449_0005_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101449_0005_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101449_0005_V,26_05_29,DJI_101449_0005_V,ok,False,None,None,None,nueva_candidata
3,DJI_20260529101506_0006_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101506_0006_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101506_0006_V,26_05_29,DJI_101506_0006_V,ok,False,None,None,None,nueva_candidata
4,DJI_20260529101533_0007_V.JPG,20260206_Geosupport_primera entrega\SOLO_JPG_0...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101533_0007_V...,CL_MLP_PAO_IF_Ortho_26_05_29_DJI_101533_0007_V,26_05_29,DJI_101533_0007_V,ok,False,None,None,None,nueva_candidata
...,...,...,...,...,...,...,...,...,...,...,...,...
1994,GEOSP-TRN-002652_GS_ORTOFOTO_EV2_170526.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EV2.tif,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EV2,26_05_17,GS_ORTOFOTO_EV2,ok,False,None,None,None,nueva_candidata
1995,GEOSP-TRN-002655_GS_ORTOFOTO_EDT_17-05-2026.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EDT.tif,CL_MLP_PAO_IF_Ortho_26_05_17_GS_ORTOFOTO_EDT,26_05_17,GS_ORTOFOTO_EDT,ok,False,None,None,None,nueva_candidata
1996,GEOSP-TRN-002656_GD_ORTOFOTO_Tramo 2 linea 22 ...,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_14_GD_ORTOFOTO_TRAMO...,CL_MLP_PAO_IF_Ortho_26_05_14_GD_ORTOFOTO_TRAMO...,26_05_14,GD_ORTOFOTO_TRAMO_2_LINEA_22_KV_E_35_E48,ok,False,None,None,None,nueva_candidata
1997,GEOSP-TRN-002660_GS_ORTOFOTO_EM1_150526.tif,20260519_Geosupport\SOLO_TIF_19May26\GEOSP-TRN...,CL_MLP_PAO_IF_Ortho_26_05_15_GS_ORTOFOTO_EM1.tif,CL_MLP_PAO_IF_Ortho_26_05_15_GS_ORTOFOTO_EM1,26_05_15,GS_ORTOFOTO_EM1,ok,False,None,None,None,nueva_candidata


## Fase 1.7 - Exportar resultados para analisis

In [38]:
def export_dataframe_csv(dataframe, output_folder, file_name):
    output_path = Path(output_folder) / file_name
    dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    return output_path


def export_dataframes_sqlite(dataframes, sqlite_path):
    import sqlite3

    sqlite_path = Path(sqlite_path)

    with sqlite3.connect(sqlite_path) as connection:
        for table_name, dataframe in dataframes.items():
            dataframe.copy().to_sql(table_name, connection, if_exists="replace", index=False)

    return sqlite_path


run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_results_dir = Path.cwd() / "outputs" / "carga_imagenes" / run_timestamp
output_results_dir.mkdir(parents=True, exist_ok=True)

result_tables = {
    "input_images": input_images_df,
    "ortho_input_images": ortho_input_images_df,
    "non_ortho_input_images": non_ortho_input_images_df,
    "mosaic_fields": mosaic_fields_df,
    "mosaic_sample": mosaic_df,
    "mosaic_image_inventory": mosaic_image_inventory_df,
    "input_expected_names": input_expected_names_df,
    "input_vs_mosaic": input_vs_mosaic_df,
    "probable_new_tifs": probable_new_tifs_df,
    "review_or_discard_tifs": review_or_discard_tifs_df,
}

if "new_images_df" in globals():
    result_tables["initial_new_images"] = new_images_df

exported_results = {
    "input_images_csv": export_dataframe_csv(input_images_df, output_results_dir, "01_input_images.csv"),
    "ortho_input_images_csv": export_dataframe_csv(ortho_input_images_df, output_results_dir, "02_ortho_input_images.csv"),
    "non_ortho_input_images_csv": export_dataframe_csv(non_ortho_input_images_df, output_results_dir, "03_non_ortho_input_images.csv"),
    "mosaic_fields_csv": export_dataframe_csv(mosaic_fields_df, output_results_dir, "04_mosaic_fields.csv"),
    "mosaic_sample_csv": export_dataframe_csv(mosaic_df, output_results_dir, "05_mosaic_sample.csv"),
    "mosaic_image_inventory_csv": export_dataframe_csv(mosaic_image_inventory_df, output_results_dir, "06_mosaic_image_inventory.csv"),
    "input_expected_names_csv": export_dataframe_csv(input_expected_names_df, output_results_dir, "07_input_expected_names.csv"),
    "input_vs_mosaic_csv": export_dataframe_csv(input_vs_mosaic_df, output_results_dir, "08_input_vs_mosaic.csv"),
    "probable_new_tifs_csv": export_dataframe_csv(probable_new_tifs_df, output_results_dir, "09_probable_new_tifs.csv"),
    "review_or_discard_tifs_csv": export_dataframe_csv(review_or_discard_tifs_df, output_results_dir, "10_review_or_discard_tifs.csv"),
}

if "new_images_df" in globals():
    exported_results["initial_new_images_csv"] = export_dataframe_csv(new_images_df, output_results_dir, "11_initial_new_images.csv")

summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "input_folder", "value": str(PATH_INPUT_SCAN)},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "input_images_count", "value": len(input_images_df)},
    {"metric": "ortho_input_images_count", "value": len(ortho_input_images_df)},
    {"metric": "non_ortho_input_images_count", "value": len(non_ortho_input_images_df)},
    {"metric": "mosaic_inventory_count", "value": len(mosaic_image_inventory_df)},
    {"metric": "expected_names_count", "value": len(input_expected_names_df)},
    {"metric": "comparison_count", "value": len(input_vs_mosaic_df)},
    {"metric": "probable_new_tifs_count", "value": len(probable_new_tifs_df)},
    {"metric": "review_or_discard_tifs_count", "value": len(review_or_discard_tifs_df)},
]

if not input_vs_mosaic_df.empty:
    for status, count in input_vs_mosaic_df["load_status"].value_counts(dropna=False).items():
        summary_rows.append({"metric": f"load_status_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)
result_tables["summary"] = summary_df
exported_results["summary_csv"] = export_dataframe_csv(summary_df, output_results_dir, "00_summary.csv")
exported_results["sqlite"] = export_dataframes_sqlite(
    result_tables,
    output_results_dir / f"carga_imagenes_{run_timestamp}.sqlite",
)

exported_results_df = pd.DataFrame(
    [{"name": name, "path": str(path)} for name, path in exported_results.items()]
)

print(f"Resultados exportados en: {output_results_dir}")
exported_results_df

Resultados exportados en: c:\Users\esrlrivero_adm\Documents\amsa-pao-geosupport\outputs\carga_imagenes\20260612_130630


,name,path
0,input_images_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
1,mosaic_fields_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
2,mosaic_sample_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
3,mosaic_image_inventory_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
4,input_expected_names_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
5,input_vs_mosaic_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
6,initial_new_images_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
7,summary_csv,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
8,sqlite,c:\Users\esrlrivero_adm\Documents\amsa-pao-geo...
